# Урок 29. Как устроены современные сервисы

10 класс · IV четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [← Урок 28](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-28.ipynb) · [Урок 30 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-30.ipynb)

---

Что происходит между нажатием кнопки и ответом на экране: клиент и сервер, запрос и ответ, API, база данных, кеш, облако. Почему приложение работает без интернета наполовину. Обзорный урок без кода.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 10А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="10-29", name=ФИО, klass=Класс)

## Что делаем

Погода, карты, мессенджеры, музыка — за каждым приложением стоят
серверы, базы данных и договорённости о том, как с ними разговаривать.
Разберём, как это устроено, и смоделируем сервис в Python: сервер,
запросы, ответы в JSON и кеш.

## Разбираемся

### Клиент и сервер

Приложение на телефоне — **клиент**. Оно почти ничего не хранит
у себя: за данными обращается к **серверу**. Клиент отправляет
**запрос** («дай погоду в Москве»), сервер отвечает **ответом** —
данными или ошибкой.

### Код ответа

Каждый ответ начинается с трёхзначного кода — по нему клиент сразу
понимает, что произошло:

| Код | Значит | Пример |
|---|---|---|
| **200** | всё хорошо, вот данные | погода пришла |
| **404** | такого нет | опечатка в адресе страницы |
| **403** | нельзя | чужой закрытый профиль |
| **500** | сломалось на сервере | ошибка в коде сервиса |

Коды на 4 — ошибка клиента (спросил не то), на 5 — ошибка сервера.

### API и JSON

**API** — правила разговора: какие запросы можно отправить и в каком
виде придёт ответ. Так погодный сервис встраивают в чужие приложения:
все пользуются одним API.

Данные обычно приходят в формате **JSON** — это текст, очень похожий
на словарь Python:

```json
{"город": "Москва", "температура": 18, "осадки": false}
```

В Python его превращают в словарь модулем `json`.

### База данных, кеш и облако

Данные сервера лежат в **базе данных**. Ходить в базу на каждый запрос
долго, поэтому ставят **кеш** — быструю временную копию самого нужного.
Кеш есть и на сервере, и на телефоне: картинки, которые ты уже видел,
второй раз не скачиваются.

Поэтому приложение частично работает без интернета: закешированное
показывается сразу, новое догружается, когда появится связь.

**Облако** — чужие компьютеры в дата-центрах, которые сервис арендует
вместо того, чтобы держать свои.

## Смотрим, как это работает

### Сервер — это функция

Получает адрес запроса, ищет данные в «базе» и возвращает код и данные.

In [ ]:
база = {
    "/погода/москва": {"город": "Москва", "температура": 18},
    "/погода/казань": {"город": "Казань", "температура": 15},
}


def сервер(адрес):
    if адрес in база:
        return 200, база[адрес]
    return 404, None


for адрес in ["/погода/москва", "/погода/мосвка"]:
    код, данные = сервер(адрес)
    print(адрес, "→", код, данные)

Опечатка в адресе — и вместо данных 404.

### JSON: текст ↔ словарь

In [ ]:
import json

ответ = '{"город": "Москва", "температура": 18, "осадки": false}'
данные = json.loads(ответ)

print(type(данные))
print("В городе", данные["город"], данные["температура"], "градусов")
print(json.dumps({"город": "Сочи", "температура": 24}, ensure_ascii=False))

`json.loads` — из текста в словарь, `json.dumps` — обратно. `false`
в JSON становится `False` в Python.

### Кеш экономит походы на сервер

In [ ]:
кеш = {}
походов = 0


def погода(адрес):
    global походов
    if адрес not in кеш:
        походов += 1
        кеш[адрес] = сервер(адрес)
    return кеш[адрес]


for адрес in ["/погода/москва", "/погода/казань", "/погода/москва", "/погода/москва"]:
    погода(адрес)

print("Запросов: 4, походов на сервер:", походов)

Четыре запроса, а на сервер сходили дважды. На миллионах пользователей
это разница между одним сервером и сотней.

## Пробуем сами

### Задача 1. Что такое клиент

Что из перечисленного чаще всего выполняет роль клиента?

In [ ]:
#@title 🧩 Задача 1. Что такое клиент { display-mode: "form" }
#@markdown Выбери ответ
ответ = "выбери ответ" #@param ["выбери ответ", "база данных", "приложение на телефоне", "облачный склад данных"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", ответ, "35ce3f42465d2deb",
         hint="Тот, кто отправляет запросы.")

### Задача 2. Что такое кеш

Как называется быстрая временная копия часто нужных данных?

In [ ]:
#@title 🧩 Задача 2. Что такое кеш { display-mode: "form" }
#@markdown Впиши слово
ответ = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", ответ, "74d8cdfd4843f0dd",
         hint="Его ещё называют «кэш».")

### Задача 3. Почему работает без связи

Почему приложение частично работает без интернета?

In [ ]:
#@title 🧩 Задача 3. Почему работает без связи { display-mode: "form" }
#@markdown Выбери ответ
ответ = "выбери ответ" #@param ["выбери ответ", "потому что сервер не нужен", "потому что часть данных закеширована на устройстве", "потому что интернет не нужен никогда"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("3", ответ, "fd2bfa4b61b43be6",
         hint="Что уже скачано, то и видно.")

### Задача 4. Свой сервер

Напиши функцию-сервер: она получает адрес и словарь-базу. Если адрес
есть в базе, возвращает пару `(200, данные)`, иначе `(404, None)`.

In [ ]:
def обработать(адрес, база):
    return ...

In [ ]:
si.check("4", обработать, [
    (("/a", {"/a": 1}), (200, 1)),
    (("/b", {"/a": 1}), (404, None)),
    (("/", {}), (404, None)),
])

### Задача 5. Разбираем JSON

Сервер погоды прислал ответ строкой JSON. Функция возвращает
температуру из поля `"temp"`.

In [ ]:
import json


def температура(ответ):
    return ...

In [ ]:
si.check("5", температура, [
    ('{"city": "Moscow", "temp": 18}', 18),
    ('{"temp": -5, "wind": 3}', -5),
])

### Задача 6. Сколько раз сходили на сервер

Функция получает список запросов (адресов) по порядку и возвращает,
сколько раз пришлось обратиться к серверу, если всё, что уже
получали, берётся из кеша.

In [ ]:
def походов_на_сервер(запросы):
    return ...

In [ ]:
si.check("6", походов_на_сервер, [
    (["a", "b", "a", "a", "c"], 3),
    ([], 0),
    (["x", "x", "x"], 1),
])

### Вопрос 7. Код 404

Сервер ответил кодом 404. Что это значит?

In [ ]:
#@title 🧩 Вопрос 7. Код 404 { display-mode: "form" }
#@markdown Выбери ответ
ответ = "выбери ответ" #@param ["выбери ответ", "всё хорошо", "такой страницы или данных нет", "сервер сломался"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("7", ответ, "4d73577b2e77e1df",
         hint="Коды на 4 — клиент спросил не то.")

---

## Домашнее задание

### 1. Разбери знакомый сервис

Выбери один сервис (музыка, карты, мессенджер) и опиши:

1. что делает клиент и что — сервер;
2. какие данные хранятся в базе;
3. что, скорее всего, закешировано на телефоне — проверь: включи
   авиарежим и посмотри, что осталось доступно;
4. какой ответ мог бы прийти в JSON на один запрос — придумай
   и запиши.

Сдаётся показом учителю.

### 2. Журнал сервера

Функция получает список кодов ответов из журнала сервера и возвращает
словарь «код → сколько раз встретился».

In [ ]:
def по_кодам(коды):
    return ...

In [ ]:
si.check("дз2", по_кодам, [
    ([200, 404, 200, 500, 200], {200: 3, 404: 1, 500: 1}),
    ([], {}),
])

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 28](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-28.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 30 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-30.ipynb)